In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import warnings
import tensorflow as tf
import pandas as pd

tf.get_logger().setLevel("ERROR")
tf.autograph.set_verbosity(0)

ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import run_unet_mlp_uv15 as base
import run_unet_mlp_bottleneckvqc_uv15_randomsplit as exp

from functions.nb_helpers import (
    signed_log1p,
    transform_y_signed_log1p,
    inverse_transform_y_signed_log1p,
    repo_root,
    resolve_extracted_uv_dir,
    resolve_weights_path,
    evaluate_split_physical,
    print_region_metrics,
    collect_grad_rows,
    plot_uv_threeway,
    plot_uv_qubits_compare,
    plot_uv_layers_compare,
    extract_valid_points_by_comp,
    shared_axis_range,
    plot_scatter_reg,
)

# Back-compat aliases
_collect_grad_rows = collect_grad_rows
_extract_valid_points_by_comp = extract_valid_points_by_comp
_shared_axis_range = shared_axis_range
_plot_scatter_reg = plot_scatter_reg

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
# Config
height_m = 15.0
last_k = 12
epochs = 1000
batch_size = 2
cond_emb_dim = 16
n_qubits = 5
n_layers = 2
seed = 7
train_frac = 0.8
val_frac = 0.1

base.set_seeds(seed)


In [ ]:
here = str(repo_root())
os.chdir(here)

extracted_uv_dir = str(resolve_extracted_uv_dir(here))

cases = base.list_cases(extracted_uv_dir)

xs, cs, ys, meta = [], [], [], []
for c in cases:
    m_building, u_mean, v_mean = base.load_uv_steady_mean(c.path, height_m=height_m, last_k=last_k)
    x = m_building[..., None].astype(np.float32)
    y = np.stack([u_mean, v_mean], axis=-1).astype(np.float32)
    xs.append(x)
    cs.append(base.cond_vector(c.speed, c.angle_deg))
    ys.append(y)
    meta.append({"file": os.path.basename(c.path), "speed": c.speed, "d_code": c.d_code, "angle_deg": c.angle_deg})

X = np.stack(xs, axis=0)
C = np.stack(cs, axis=0)
Y = np.stack(ys, axis=0)

X.shape, C.shape, Y.shape


In [ ]:
# Non-building u_mean / v_mean distributions
u_all = Y[..., 0]
v_all = Y[..., 1]

valid_mask = (u_all != base.MISSING_VALUE) & (v_all != base.MISSING_VALUE)
u_valid = u_all[valid_mask]
v_valid = v_all[valid_mask]

# Signed log1p transform: x -> sign(x) * log1p(|x|)

u_log = signed_log1p(u_valid)
v_log = signed_log1p(v_valid)

# Train targets in signed-log1p space (buildings keep missing value)
Y_log = transform_y_signed_log1p(Y, base.MISSING_VALUE)

In [ ]:
# random split
if train_frac + val_frac >= 1.0:
    raise ValueError(f"train_frac+val_frac must be < 1. Got {train_frac+val_frac}")

split_idx = exp._random_split_indices(n=len(cases), train_frac=train_frac, val_frac=val_frac, seed=seed)
tr, va, te = split_idx["train"], split_idx["val"], split_idx["test"]

# Condition scaling (fit on train only)
c_scaler = base.StandardScaler()
C_tr = c_scaler.fit_transform(C[tr])
C_va = c_scaler.transform(C[va])
C_te = c_scaler.transform(C[te])

# Output normalization in signed-log1p space (fit on train only, ignoring missing)
y_mean, y_std = base.compute_y_norm_stats(Y_log[tr])
Y_tr = base.normalize_y(Y_log[tr], y_mean, y_std)
Y_va = base.normalize_y(Y_log[va], y_mean, y_std)
Y_te = base.normalize_y(Y_log[te], y_mean, y_std)

X_tr, X_va, X_te = X[tr], X[va], X[te]
len(tr), len(va), len(te)


# C-QB-UNet

In [ ]:
# # Full epochs training
# model = exp.build_unet_cond_mlp_bottleneck_vqc(
#     input_shape=X_tr.shape[1:],
#     cond_dim=C_tr.shape[-1],
#     cond_emb_dim=cond_emb_dim,
#     n_qubits=n_qubits,
#     n_layers=n_layers,
# )

# model.compile(
#     optimizer=base.tf.keras.optimizers.Adam(1e-3),
#     loss=base.masked_mse_with_grad,
#     metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
# )

# checkpoint_dir = os.path.join(here, "checkpoints", "bottleneckvqc_unet_log1p")
# os.makedirs(checkpoint_dir, exist_ok=True)
# best_weights_path = os.path.join(checkpoint_dir, "best_val_weights.h5")

# callbacks = [
#     base.tf.keras.callbacks.ModelCheckpoint(
#         filepath=best_weights_path,
#         monitor="val_loss",
#         save_best_only=True,
#         save_weights_only=True,
#         mode="min",
#         verbose=1,
#     ),
#     base.tf.keras.callbacks.ReduceLROnPlateau(
#         monitor="val_loss", factor=0.5, patience=10, min_lr=1e-5, verbose=1
#     ),
# ]

# history = model.fit(
#     x={"mask_img": X_tr, "cond": C_tr},
#     y=Y_tr,
#     validation_data=({"mask_img": X_va, "cond": C_va}, Y_va),
#     epochs=epochs,
#     batch_size=batch_size,
#     callbacks=callbacks,
#     verbose=1,
# )

# # Load best val_loss weights after training
# model.load_weights(best_weights_path)


In [ ]:
# load pretrained weights
save_dir = os.path.join(here, "checkpoints", "bottleneckvqc_unet_log1p")
os.makedirs(save_dir, exist_ok=True)
best_weights_path = os.path.join(save_dir, "best_val_weights_5q2l.h5")

# Reload model and stats (rebuild the same architecture, then load weights)
model = exp.build_unet_cond_mlp_bottleneck_vqc(
    input_shape=X_tr.shape[1:],
    cond_dim=C_tr.shape[-1],
    cond_emb_dim=cond_emb_dim,
    n_qubits=n_qubits,
    n_layers=n_layers,
)

model.compile(
    optimizer=base.tf.keras.optimizers.Adam(1e-3),
    loss=base.masked_mse_with_grad,
    metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
)

model.load_weights(best_weights_path)


In [ ]:
print("Train/Val/Test metrics (evaluated in m/s after inverse transform):")

Y_tr_denorm, pred_tr_denorm, metrics_tr = evaluate_split_physical(model, "train", X_tr, C_tr, Y_tr, y_mean, y_std)
Y_va_denorm, pred_va_denorm, metrics_va = evaluate_split_physical(model, "val", X_va, C_va, Y_va, y_mean, y_std)
Y_te_denorm, pred_te_denorm, metrics_te = evaluate_split_physical(model, "test", X_te, C_te, Y_te, y_mean, y_std)

In [ ]:
# Region-wise metrics: top-left open area vs the remaining area
# You can tune these two values to match your "left-top open area" definition.
tl_h, tl_w = 75, 90

# Build region masks from current output size

print("Region-wise metrics (m/s):")
print_region_metrics("train", Y_tr_denorm, pred_tr_denorm, tl_h=tl_h, tl_w=tl_w)
print_region_metrics("val", Y_va_denorm, pred_va_denorm, tl_h=tl_h, tl_w=tl_w)
print_region_metrics("test", Y_te_denorm, pred_te_denorm, tl_h=tl_h, tl_w=tl_w)


# C-UNet

In [ ]:
# # Full epochs training

# # C-UNet config (reuse same data/split)
# mlp_epochs = epochs
# mlp_batch_size = batch_size

# model_mlp = base.build_unet_cond(input_shape=X_tr.shape[1:], cond_dim=C_tr.shape[-1])
# model_mlp.compile(
#     optimizer=base.tf.keras.optimizers.Adam(1e-3),
#     loss=base.masked_mse_with_grad,
#     metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
# )

# checkpoint_dir = os.path.join(here, "checkpoints", "mlp_unet_log1p")
# os.makedirs(checkpoint_dir, exist_ok=True)
# best_weights_path = os.path.join(checkpoint_dir, "best_val_weights.h5")

# callbacks_mlp = [
#     base.tf.keras.callbacks.ModelCheckpoint(
#         filepath=best_weights_path,
#         monitor="val_loss",
#         save_best_only=True,
#         save_weights_only=True,
#         mode="min",
#         verbose=1,
#     ),
#     base.tf.keras.callbacks.ReduceLROnPlateau(
#         monitor="val_loss", factor=0.5, patience=10, min_lr=1e-5, verbose=1
#     ),
# ]

# history_mlp = model_mlp.fit(
#     x={"mask_img": X_tr, "cond": C_tr},
#     y=Y_tr,
#     validation_data=({"mask_img": X_va, "cond": C_va}, Y_va),
#     epochs=mlp_epochs,
#     batch_size=mlp_batch_size,
#     callbacks=callbacks_mlp,
#     verbose=1,
# )

# # Load best val_loss weights after training
# model_mlp.load_weights(best_weights_path)

In [ ]:
# load pretrained weights
save_dir = os.path.join(here, "checkpoints", "mlp_unet_log1p")
os.makedirs(save_dir, exist_ok=True)
best_weights_path = os.path.join(save_dir, "best_val_weights.h5")

# Reload model and stats (rebuild the same architecture, then load weights)
model_mlp = base.build_unet_cond(input_shape=X_tr.shape[1:], cond_dim=C_tr.shape[-1])

model_mlp.compile(
    optimizer=base.tf.keras.optimizers.Adam(1e-3),
    loss=base.masked_mse_with_grad,
    metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
)

model_mlp.load_weights(best_weights_path)


In [ ]:
print("C-UNet Train/Val/Test metrics (evaluated in m/s after inverse transform):")

Y_tr_denorm_mlp, pred_tr_denorm_mlp, metrics_tr_mlp = evaluate_split_physical(model_mlp, "train", X_tr, C_tr, Y_tr, y_mean, y_std)
Y_va_denorm_mlp, pred_va_denorm_mlp, metrics_va_mlp = evaluate_split_physical(model_mlp, "val", X_va, C_va, Y_va, y_mean, y_std)
Y_te_denorm_mlp, pred_te_denorm_mlp, metrics_te_mlp = evaluate_split_physical(model_mlp, "test", X_te, C_te, Y_te, y_mean, y_std)


In [ ]:
print("Region-wise metrics (m/s):")
print_region_metrics("train", Y_tr_denorm_mlp, pred_tr_denorm_mlp)
print_region_metrics("val", Y_va_denorm_mlp, pred_va_denorm_mlp)
print_region_metrics("test", Y_te_denorm_mlp, pred_te_denorm_mlp)


# U/V Spatial Field

In [ ]:
show_ids = [0, 1, 2]
for i in show_ids:
    m = meta[te[i]]
    plot_uv_threeway(
        Y_te_denorm,
        pred_te_denorm,
        pred_te_denorm_mlp,
        i,
        title=f"U/V Field Comparison ({m['speed']}m/s {m['angle_deg']}deg)",
    )


# N_qubits

In [ ]:
# Predictions vs qubits (3/4/5/6/7) on the test set
# Columns: Truth + 3/4/5/6/7 qubits; rows: U / V

qubits_list = [3, 4, 5, 6, 7]
# 5 qubits uses best_val_weights_5q2l; others use best_val_weights_{q}
ckpt_stem_map = {
    3: "best_val_weights_3qubits",
    4: "best_val_weights_4qubits",
    5: "best_val_weights_5q2l",
    6: "best_val_weights_6qubits",
    7: "best_val_weights_7qubits",
}

ckpt_root = os.path.join(here, "checkpoints", "bottleneckvqc_unet_log1p")

def pred_test_denorm_for_qubits(nq: int):
    model_q = exp.build_unet_cond_mlp_bottleneck_vqc(
        input_shape=X_tr.shape[1:],
        cond_dim=C_tr.shape[-1],
        cond_emb_dim=cond_emb_dim,
        n_qubits=nq,
        n_layers=n_layers,
    )
    model_q.compile(
        optimizer=base.tf.keras.optimizers.Adam(1e-3),
        loss=base.masked_mse_with_grad,
        metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
    )
    stem = ckpt_stem_map[nq]
    w_path = resolve_weights_path(ckpt_root, stem)
    model_q.load_weights(w_path)
    pred_norm = model_q.predict({"mask_img": X_te, "cond": C_te}, verbose=0).astype(np.float32)
    pred_log = base.denormalize_y(pred_norm, y_mean, y_std)
    pred_denorm = inverse_transform_y_signed_log1p(pred_log, base.MISSING_VALUE)
    print(f"loaded q={nq} from: {w_path}")
    return pred_denorm

pred_te_denorm_by_q = {}
for q in qubits_list:
    pred_te_denorm_by_q[q] = pred_test_denorm_for_qubits(q)


In [ ]:
# Change this to another test sample index if needed
sample_idx = 0
plot_uv_qubits_compare(Y_te_denorm, pred_te_denorm_by_q, idx=sample_idx)

In [ ]:
# Evaluate MAE / RMSE / R2 across qubit counts on test

# Reuse pred_te_denorm_by_q ({q: pred_array}) if already computed
# y_true is test ground truth in physical m/s
y_true = Y_te_denorm

rows = []
for q in sorted(pred_te_denorm_by_q.keys()):
    y_pred = pred_te_denorm_by_q[q]
    m = base.evaluate_numpy(y_true, y_pred)  # typically returns mae/rmse/r2/mse
    
    rows.append({
        "qubits": q,
        "MAE": m.get("mae", np.nan),
        "RMSE": m.get("rmse", np.nan),
        "R2": m.get("r2", np.nan),
    })

df_metrics_qubits = pd.DataFrame(rows).sort_values("qubits").reset_index(drop=True)
print(df_metrics_qubits)

# Optional: save
# df_metrics_qubits.to_csv("qubits_test_metrics.csv", index=False)


# N_layers

In [ ]:
# Predictions vs layers (1/2/3/4/5) on the test set
# Columns: Truth + 1/2/3/4/5 layers; rows: U / V

layers_list = [1, 2, 3, 4, 5]
# 2 layers uses best_val_weights_5q2l; others use best_val_weights_{q}
ckpt_stem_map = {
    1: "best_val_weights_1layer",
    2: "best_val_weights_5q2l",
    3: "best_val_weights_3layer",
    4: "best_val_weights_4layer",
    5: "best_val_weights_5layer",
}

ckpt_root = os.path.join(here, "checkpoints", "bottleneckvqc_unet_log1p")

def pred_test_denorm_for_qubits(nq: int):
    model_q = exp.build_unet_cond_mlp_bottleneck_vqc(
        input_shape=X_tr.shape[1:],
        cond_dim=C_tr.shape[-1],
        cond_emb_dim=cond_emb_dim,
        n_qubits=n_qubits,
        n_layers=nq,
    )
    model_q.compile(
        optimizer=base.tf.keras.optimizers.Adam(1e-3),
        loss=base.masked_mse_with_grad,
        metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
    )
    stem = ckpt_stem_map[nq]
    w_path = resolve_weights_path(ckpt_root, stem)
    model_q.load_weights(w_path)
    pred_norm = model_q.predict({"mask_img": X_te, "cond": C_te}, verbose=0).astype(np.float32)
    pred_log = base.denormalize_y(pred_norm, y_mean, y_std)
    pred_denorm = inverse_transform_y_signed_log1p(pred_log, base.MISSING_VALUE)
    print(f"loaded q={nq} from: {w_path}")
    return pred_denorm

pred_te_denorm_by_q = {}
for q in layers_list:
    pred_te_denorm_by_q[q] = pred_test_denorm_for_qubits(q)


In [ ]:
# Change this to another test sample index if needed
sample_idx = 0
plot_uv_layers_compare(Y_te_denorm, pred_te_denorm_by_q, idx=sample_idx)


In [ ]:
# Evaluate MAE / RMSE / R2 across layer counts on test

# Reuse pred_te_denorm_by_q ({q: pred_array}) if already computed
# y_true is test ground truth in physical m/s
y_true = Y_te_denorm

rows = []
for q in sorted(pred_te_denorm_by_q.keys()):
    y_pred = pred_te_denorm_by_q[q]
    m = base.evaluate_numpy(y_true, y_pred)  # typically returns mae/rmse/r2/mse
    
    rows.append({
        "layers": q,
        "MAE": m.get("mae", np.nan),
        "RMSE": m.get("rmse", np.nan),
        "R2": m.get("r2", np.nan),
    })

df_metrics_qubits = pd.DataFrame(rows).sort_values("layers").reset_index(drop=True)
print(df_metrics_qubits)

# Optional: save
# df_metrics_qubits.to_csv("qubits_test_metrics.csv", index=False)


# Gradient Loss

In [ ]:
# Gradient errors of model / model_mlp on train/val/test (physical m/s)

rows = []

# model (C-QB-UNet)
rows += _collect_grad_rows('model', 'train', Y_tr_denorm, pred_tr_denorm)
rows += _collect_grad_rows('model', 'val',   Y_va_denorm, pred_va_denorm)
rows += _collect_grad_rows('model', 'test',  Y_te_denorm, pred_te_denorm)

# model_mlp (C-UNet)
rows += _collect_grad_rows('model_mlp', 'train', Y_tr_denorm_mlp, pred_tr_denorm_mlp)
rows += _collect_grad_rows('model_mlp', 'val',   Y_va_denorm_mlp, pred_va_denorm_mlp)
rows += _collect_grad_rows('model_mlp', 'test',  Y_te_denorm_mlp, pred_te_denorm_mlp)

df_grad_metrics = pd.DataFrame(rows)
order_model = {'model': 0, 'model_mlp': 1}
order_split = {'train': 0, 'val': 1, 'test': 2}
order_comp = {'ALL': 0, 'U': 1, 'V': 2}
df_grad_metrics = (
    df_grad_metrics
    .assign(
        _m=df_grad_metrics['model'].map(order_model),
        _s=df_grad_metrics['split'].map(order_split),
        _c=df_grad_metrics['component'].map(order_comp),
    )
    .sort_values(['_m', '_s', '_c'])
    .drop(columns=['_m', '_s', '_c'])
    .reset_index(drop=True)
)

print(df_grad_metrics)

# Optional save
# df_grad_metrics.to_csv('grad_metrics_model_vs_model_mlp.csv', index=False)


In [ ]:
# 2x2 regression scatter (all valid test pixels)
# Rows: U / V; columns: C-UNet / C-QB-UNet
# Shared axis range per component (U shared, V shared)

# Test set: C-QB-UNet (model) vs C-UNet (model_mlp)
# comp_idx: 0=U, 1=V
u_true_mlp, u_pred_mlp = _extract_valid_points_by_comp(Y_te_denorm_mlp, pred_te_denorm_mlp, 0, base.MISSING_VALUE)
u_true_qb,  u_pred_qb  = _extract_valid_points_by_comp(Y_te_denorm,     pred_te_denorm,     0, base.MISSING_VALUE)

v_true_mlp, v_pred_mlp = _extract_valid_points_by_comp(Y_te_denorm_mlp, pred_te_denorm_mlp, 1, base.MISSING_VALUE)
v_true_qb,  v_pred_qb  = _extract_valid_points_by_comp(Y_te_denorm,     pred_te_denorm,     1, base.MISSING_VALUE)

# Shared axes: both U panels share one range; both V panels share one
u_axis = _shared_axis_range(u_true_mlp, u_pred_mlp, u_true_qb, u_pred_qb)
v_axis = _shared_axis_range(v_true_mlp, v_pred_mlp, v_true_qb, v_pred_qb)

fig, axes = plt.subplots(2, 2, figsize=(7, 6.3), constrained_layout=True)

stat_u_mlp = _plot_scatter_reg(axes[0, 0], u_true_mlp, u_pred_mlp, 'U (C-UNet)', axis_range=u_axis, point_color='tab:blue')
stat_u_qb  = _plot_scatter_reg(axes[0, 1], u_true_qb,  u_pred_qb,  'U (C-QB-UNet)', axis_range=u_axis, point_color='tab:orange')
stat_v_mlp = _plot_scatter_reg(axes[1, 0], v_true_mlp, v_pred_mlp, 'V (C-UNet)', axis_range=v_axis, point_color='tab:blue')
stat_v_qb  = _plot_scatter_reg(axes[1, 1], v_true_qb,  v_pred_qb,  'V (C-QB-UNet)', axis_range=v_axis, point_color='tab:orange')

fig.suptitle('Scatter Plots of Pointwise Velocity Predictions', fontsize=13)
plt.show()

# Per-component R2 for comparison with overall R2
print('Component-wise R2 on test set (pointwise):')
print(f"C-UNet    U-R2={stat_u_mlp['r2']:.6f}, V-R2={stat_v_mlp['r2']:.6f}")
print(f"C-QB-UNet U-R2={stat_u_qb['r2']:.6f}, V-R2={stat_v_qb['r2']:.6f}")
